In [32]:
import collections
import re

from d2l import torch as d2l

In [33]:
class TimeMachine(d2l.DataModule):
    """The Time Machine dataset."""

    def _download(self):
        filename = d2l.download(
            d2l.DATA_URL + "timemachine.txt",
            self.root,
            "090b5e7e70c295757f55df93cb0a180b9691891a",
        )

        with open(filename) as file:
            return file.read()
        
    def _preprocess(self, text):
        return re.sub(
            "[^A-Za-z]+",
            " ",
            text,   
        ).lower()
        
    def _tokenize(self, text):
        return list(text)

In [ ]:
# Preprocessing & Tokenization

data = TimeMachine()

raw_text = data._download()
text = data._preprocess(raw_text)
tokens = data._tokenize(text)

tokens[:10]

['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']

In [ ]:
class Vocab:
    """Vocabulary for text."""
    
    def __init__(
        self,
        tokens=[],
        min_freq=0,
        reserved_tokens=[],
    ):
        
        # element type이 list이면 Flatten
        if tokens and isinstance(tokens[0], list):
            tokens = [
                token
                for line in tokens
                for token in line
            ]
            
        # {
        #     "a": 3,
        #     "b": 2,
        #     "c": 1,
        # }
        counter = collections.Counter(tokens)
            
        # 각 Tuple에서 두 번째 값(frequency)으로 Reverse 정렬
        self.token_freqs = sorted(
            counter.items(),
            key=lambda item: item[1], 
            reverse=True,
        )
        
        # index -> token Mapping
        self.idx_to_token = list(
            sorted(
                set(
                    ["<unk>"]
                    + reserved_tokens
                    + [
                        token
                        for token, freq in self.token_freqs
                        if freq >= min_freq
                    ]
                )
            )
        )
        
        # token -> index Mapping
        self.token_to_idx = {
            token: index
            for index, token in enumerate(self.idx_to_token)
        }
        
        
    def __len__(self):
        return len(self.idx_to_token)
    
    def __getitem__(self, tokens):
        
        # 입력이 token 하나면,
        if not isinstance(tokens, (list, tuple)):
            return self.token_to_idx.get(
                tokens,
                self.unk,
            )

        # 입력이 list, tuple이라면,
        return [
            self.__getitem__(token)
            for token in tokens
        ]

    def to_tokens(self, indices):
        if hasattr(indices, "__len__") and len(indices) > 1:
            return [
                self.idx_to_token[int(index)]
                for index in indices
            ]

        return self.idx_to_token[indices]

    @property
    def unk(self):
        return self.token_to_idx["<unk>"]

In [ ]:
# Numericalize

vocab = Vocab(tokens)
indices = vocab[tokens[:10]]

print(
    "indices:",
    indices,
)

print(
    "words:",
    vocab.to_tokens(indices),
)

indices: [21, 9, 6, 0, 21, 10, 14, 6, 0, 14]
words: ['t', 'h', 'e', ' ', 't', 'i', 'm', 'e', ' ', 'm']


In [ ]:
def build(
    self,
    raw_text,
    vocab=None,
):
    """
    <Putting It All Together>
    1) Preprocessing
    2) Tokenize
    3) Numericalize
    """
    
    tokens = self._tokenize(
        self._preprocess(raw_text)
    )
    
    if vocab is None:
        vocab = Vocab(tokens)
        
    corpus = [
        vocab[token]
        for token in tokens
    ]
    
    return corpus, vocab

TimeMachine.build = build

In [51]:
corpus, vocab = data.build(raw_text)

print("len(corpus):", len(corpus))
print("len(vocab):", len(vocab))

print("corpus[:20]", corpus[:20])

len(corpus): 173428
len(vocab): 28
corpus[:20] [21, 9, 6, 0, 21, 10, 14, 6, 0, 14, 2, 4, 9, 10, 15, 6, 0, 3, 26, 0]
